# LNG Fleet Portfolio Optimization
### Six-Vessel Fleet Simulation

*Deterministic route matrix · FOB/DES contract logic · initial hedging · correlated stochastic modelling · mean-reverting freight · two-stage optimization*

---
**Scope:** US supply priced off Henry Hub · West Africa and Pacific priced off Brent-linked formulas · sales into Europe (TTF) and Asia (JKM)

## 0. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False

print('All libraries loaded successfully.')

---
## Step 1 — Market Assumptions & FOB Formulas

In [ ]:
# ── MARKET BENCHMARKS (current spot) ─────────────────────────────────────────
BRENT = 82.0    # USD/bbl
HH    =  2.4    # USD/MMBtu  Henry Hub
TTF   = 11.5    # USD/MMBtu  European gas
JKM   = 13.2    # USD/MMBtu  Japan Korea Marker

# ── SUPPLY PRICING ───────────────────────────────────────────────────────────
C_US              = 3.0    # USD/MMBtu  liquefaction + fixed uplift
WAFR_BRENT_FACTOR = 0.08   # West Africa: 8% of Brent
PAC_BRENT_FACTOR  = 0.12   # Pacific:     12% of Brent

FOB_US   = HH + C_US
FOB_WAFR = WAFR_BRENT_FACTOR * BRENT
FOB_PAC  = PAC_BRENT_FACTOR  * BRENT
FOB      = {'US': FOB_US, 'W.Africa': FOB_WAFR, 'Pacific': FOB_PAC}

# ── DESTINATION PRICES ───────────────────────────────────────────────────────
DEST_PRICE = {'Europe': TTF, 'Asia': JKM}

# ── VOYAGE DURATIONS (round-trip days incl. turnaround) ──────────────────────
ROUTE_DAYS = {
    ('US',       'Europe'): 28,
    ('US',       'Asia'):   52,
    ('W.Africa', 'Europe'): 18,
    ('W.Africa', 'Asia'):   38,
    ('Pacific',  'Europe'): 60,
    ('Pacific',  'Asia'):   22,
}
LOADED_DAY_FACTOR = 0.50

# ── SHIPPING COST COMPONENTS (USD/MMBtu, at freight index = 1.0) ─────────────
BASE_FREIGHT = {
    ('US',       'Europe'): 0.95,
    ('US',       'Asia'):   1.35,
    ('W.Africa', 'Europe'): 0.55,
    ('W.Africa', 'Asia'):   1.00,
    ('Pacific',  'Europe'): 1.55,
    ('Pacific',  'Asia'):   0.45,
}
PORT_COST = {
    ('US',       'Europe'): 0.16, ('US',       'Asia'): 0.18,
    ('W.Africa', 'Europe'): 0.14, ('W.Africa', 'Asia'): 0.16,
    ('Pacific',  'Europe'): 0.18, ('Pacific',  'Asia'): 0.12,
}
CANAL_COST = {
    ('US',       'Europe'): 0.06, ('US',       'Asia'): 0.24,
    ('W.Africa', 'Europe'): 0.02, ('W.Africa', 'Asia'): 0.12,
    ('Pacific',  'Europe'): 0.20, ('Pacific',  'Asia'): 0.02,
}
FREIGHT_BETA = {
    ('US',       'Europe'): 0.70, ('US',       'Asia'): 1.00,
    ('W.Africa', 'Europe'): 0.55, ('W.Africa', 'Asia'): 0.80,
    ('Pacific',  'Europe'): 1.10, ('Pacific',  'Asia'): 0.45,
}

# ── BOIL-OFF (fixed reference price — removes circular dependency) ────────────
BOILOFF_DAILY_RATE = 0.0010   # 0.10% of cargo per loaded day
BOILOFF_REF_PRICE  = 12.0     # USD/MMBtu fixed reference

BASE_FREIGHT_INDEX  = 1.0
DES_SERVICE_PREMIUM = 0.35
CARGO_SIZE_MMBTU    = 3_300_000

print(f'Brent: ${BRENT:.1f}/bbl  |  HH: ${HH:.2f}  |  TTF: ${TTF:.2f}  |  JKM: ${JKM:.2f}  (USD/MMBtu)')
print(f'FOB_US: ${FOB_US:.2f}  |  FOB_WAFR: ${FOB_WAFR:.2f}  |  FOB_PAC: ${FOB_PAC:.2f}  (USD/MMBtu)')

---
## Step 2 — Route Cost & Margin Functions

In [ ]:
def loaded_days(origin, destination):
    return ROUTE_DAYS[(origin, destination)] * LOADED_DAY_FACTOR

def route_cost_components(origin, destination, freight_index=BASE_FREIGHT_INDEX):
    ld      = loaded_days(origin, destination)
    boiloff = BOILOFF_DAILY_RATE * ld * BOILOFF_REF_PRICE
    freight = BASE_FREIGHT[(origin, destination)] * (
        1 + FREIGHT_BETA[(origin, destination)] * (freight_index - 1))
    port    = PORT_COST[(origin, destination)]
    canal   = CANAL_COST[(origin, destination)]
    total   = freight + boiloff + port + canal
    return {'Freight': freight, 'Boil-off': boiloff,
            'Port': port, 'Canal': canal, 'Total': total}

def fob_cost(origin, brent=None, hh=None):
    brent = BRENT if brent is None else brent
    hh    = HH    if hh    is None else hh
    if origin == 'US':       return hh + C_US
    if origin == 'W.Africa': return WAFR_BRENT_FACTOR * brent
    return PAC_BRENT_FACTOR * brent

def route_margin(origin, destination, contract='FOB',
                 brent=None, hh=None, ttf=None, jkm=None,
                 freight_index=BASE_FREIGHT_INDEX):
    brent = BRENT if brent is None else brent
    hh    = HH    if hh    is None else hh
    ttf   = TTF   if ttf   is None else ttf
    jkm   = JKM   if jkm   is None else jkm
    fob   = fob_cost(origin, brent=brent, hh=hh)
    dp    = ttf if destination == 'Europe' else jkm
    comps = route_cost_components(origin, destination, freight_index=freight_index)
    premium = DES_SERVICE_PREMIUM if contract == 'DES' else 0.0
    return dp - fob - comps['Total'] + premium

for orig in ['US', 'W.Africa', 'Pacific']:
    for dest in ['Europe', 'Asia']:
        print(f'{orig:10s} → {dest:6s}  FOB: {route_margin(orig,dest,"FOB"):+.2f}  DES: {route_margin(orig,dest,"DES"):+.2f}  (USD/MMBtu)')

---
## Step 3 — Deterministic Route Matrix

In [ ]:
origins      = ['US', 'W.Africa', 'Pacific']
destinations = ['Europe', 'Asia']

rows = []
for orig in origins:
    for dest in destinations:
        days  = ROUTE_DAYS[(orig, dest)]
        comps = route_cost_components(orig, dest)
        m_fob = route_margin(orig, dest, 'FOB')
        m_des = route_margin(orig, dest, 'DES')
        rows.append({
            'Origin': orig, 'Destination': dest,
            'FOB ($/MMBtu)':  round(FOB[orig], 2),
            'Freight':        round(comps['Freight'], 3),
            'Boil-off':       round(comps['Boil-off'], 3),
            'Port+Canal':     round(comps['Port'] + comps['Canal'], 3),
            'Total Cost':     round(comps['Total'], 3),
            'Dest Price':     DEST_PRICE[dest],
            'FOB Margin':     round(m_fob, 3),
            'DES Margin':     round(m_des, 3),
            'RT Days':        days,
            'FOB Margin/day': round(m_fob / days, 4),
            'DES Margin/day': round(m_des / days, 4),
        })

route_matrix = pd.DataFrame(rows).sort_values('FOB Margin/day', ascending=False).reset_index(drop=True)
print('=== DETERMINISTIC ROUTE MATRIX ===')
print(route_matrix.to_string(index=False))

In [ ]:
# ── Heatmaps ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
cmap = LinearSegmentedColormap.from_list('rg', ['#c0392b','#f39c12','#27ae60'])
panels = [
    ('FOB Margin',     'FOB Margin ($/MMBtu)'),
    ('DES Margin',     'DES Margin ($/MMBtu)'),
    ('FOB Margin/day', 'FOB Margin/day'),
    ('DES Margin/day', 'DES Margin/day'),
]
for ax, (col, title) in zip(axes.flatten(), panels):
    piv = route_matrix.pivot(index='Origin', columns='Destination', values=col)
    im  = ax.imshow(piv.values, cmap=cmap, aspect='auto')
    ax.set_xticks(range(len(piv.columns))); ax.set_xticklabels(piv.columns)
    ax.set_yticks(range(len(piv.index)));   ax.set_yticklabels(piv.index)
    for i in range(piv.shape[0]):
        for j in range(piv.shape[1]):
            ax.text(j, i, f'{piv.values[i,j]:.3f}', ha='center', va='center',
                    fontsize=12, fontweight='bold', color='white')
    ax.set_title(title, fontweight='bold')
    plt.colorbar(im, ax=ax)
plt.suptitle('Route Economics — FOB vs DES', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── Cost waterfall ────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 4.5))
route_labels = [f"{r['Origin'][:4]}→{r['Destination'][:3]}" for _, r in route_matrix.iterrows()]
x = np.arange(len(route_labels)); w = 0.55
ax.bar(x, route_matrix['Freight'].values, w, label='Freight',    color='#2980b9', alpha=0.9)
ax.bar(x, route_matrix['Boil-off'].values, w, bottom=route_matrix['Freight'].values,
       label='Boil-off', color='#e67e22', alpha=0.9)
ax.bar(x, route_matrix['Port+Canal'].values, w,
       bottom=route_matrix['Freight'].values + route_matrix['Boil-off'].values,
       label='Port+Canal', color='#8e44ad', alpha=0.9)
ax.set_xticks(x); ax.set_xticklabels(route_labels)
ax.set_ylabel('Cost ($/MMBtu)')
ax.set_title('Route Cost Decomposition', fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.25)
plt.tight_layout(); plt.show()

---
## Step 4 — Ship-by-Ship Fleet Matrix

In [ ]:
fleet_data = [
    {'Ship': 'Ship 1', 'Basin': 'Atlantic',  'Category': 'Committed', 'Contract': 'DES',
     'Flexible Destinations': False, 'Hedge Ratio': 0.85, 'Option Premium': 0.02,
     'Best Origin': 'US',       'Best Dest': 'Europe'},
    {'Ship': 'Ship 2', 'Basin': 'Atlantic',  'Category': 'Swing',     'Contract': 'FOB',
     'Flexible Destinations': True,  'Hedge Ratio': 0.45, 'Option Premium': 0.30,
     'Best Origin': 'W.Africa', 'Best Dest': 'Asia'},
    {'Ship': 'Ship 3', 'Basin': 'Iberia',    'Category': 'Committed', 'Contract': 'DES',
     'Flexible Destinations': False, 'Hedge Ratio': 0.80, 'Option Premium': 0.03,
     'Best Origin': 'W.Africa', 'Best Dest': 'Europe'},
    {'Ship': 'Ship 4', 'Basin': 'Pacific',   'Category': 'Committed', 'Contract': 'DES',
     'Flexible Destinations': False, 'Hedge Ratio': 0.90, 'Option Premium': 0.02,
     'Best Origin': 'Pacific',  'Best Dest': 'Asia'},
    {'Ship': 'Ship 5', 'Basin': 'W.Africa',  'Category': 'Swing',     'Contract': 'FOB',
     'Flexible Destinations': True,  'Hedge Ratio': 0.40, 'Option Premium': 0.35,
     'Best Origin': 'W.Africa', 'Best Dest': 'Europe'},
    {'Ship': 'Ship 6', 'Basin': 'US Gulf',   'Category': 'Swing',     'Contract': 'FOB',
     'Flexible Destinations': True,  'Hedge Ratio': 0.50, 'Option Premium': 0.20,
     'Best Origin': 'US',       'Best Dest': 'Europe'},
]

fleet = pd.DataFrame(fleet_data)
fleet['Effective Option Premium'] = np.where(
    (fleet['Contract'] == 'FOB') & fleet['Flexible Destinations'],
    fleet['Option Premium'], 0.0)

def get_economics(row):
    m   = route_margin(row['Best Origin'], row['Best Dest'], row['Contract'])
    day = ROUTE_DAYS[(row['Best Origin'], row['Best Dest'])]
    return pd.Series([m, m/day, day])

fleet[['Base Margin','Margin/day','RT Days']] = fleet.apply(get_economics, axis=1)
fleet['Adj Margin/day'] = fleet['Margin/day'] + fleet['Effective Option Premium'] / fleet['RT Days']
fleet['Action'] = fleet.apply(lambda r: (
    f"Fix {int(r['Hedge Ratio']*100)}% → {r['Best Dest']} (locked)"
    if r['Contract'] == 'DES' or not r['Flexible Destinations']
    else f"Retain optionality; hedge {int(r['Hedge Ratio']*100)}%, default {r['Best Dest']}"
), axis=1)

print('=== FLEET MATRIX ===')
print(fleet[['Ship','Basin','Category','Contract','Flexible Destinations',
             'Best Origin','Best Dest','Base Margin','Margin/day',
             'Effective Option Premium','Hedge Ratio','Action']].to_string(index=False))

In [ ]:
colors = {'Committed': '#2980b9', 'Swing': '#e67e22'}
bar_colors = [colors[c] for c in fleet['Category']]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

ax = axes[0]
ax.barh(fleet['Ship'], fleet['Adj Margin/day'], color=bar_colors, alpha=0.88)
ax.set_xlabel('Adj. Margin/day ($/MMBtu/day)')
ax.set_title('Time-adjusted economics', fontweight='bold')
for i, (_, row) in enumerate(fleet.iterrows()):
    ax.text(row['Adj Margin/day']+0.001, i, f"{row['Contract']}/{row['Category'][:4]}",
            va='center', fontsize=8)
ax.grid(axis='x', alpha=0.25)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=v, label=k) for k,v in colors.items()], fontsize=9)

ax2 = axes[1]
ax2.scatter(fleet['RT Days'], fleet['Option Premium'], c=bar_colors, s=120, alpha=0.9, zorder=3)
for _, row in fleet.iterrows():
    ax2.annotate(row['Ship'], (row['RT Days'], row['Option Premium']),
                 textcoords='offset points', xytext=(6,4), fontsize=8)
ax2.set_xlabel('Round-trip days'); ax2.set_ylabel('Option Premium ($/MMBtu)')
ax2.set_title('Option premium vs voyage length', fontweight='bold')
ax2.grid(alpha=0.25)
plt.suptitle('Fleet — Ship-Level Economics', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

---
## Step 5 — Initial Hedging Layer

In [ ]:
hedge_rows = []
for _, ship in fleet.iterrows():
    ratio  = ship['Hedge Ratio']
    locked = ship['Base Margin'] * ratio * CARGO_SIZE_MMBTU / 1e6
    open_  = ship['Base Margin'] * (1-ratio) * CARGO_SIZE_MMBTU / 1e6
    benchmark = ('HH+' if ship['Best Origin']=='US' else 'Brent+') + \
                ('TTF' if ship['Best Dest']=='Europe' else 'JKM')
    hedge_rows.append({
        'Ship': ship['Ship'], 'Category': ship['Category'], 'Contract': ship['Contract'],
        'Hedge Ratio': f"{ratio*100:.0f}%",
        'Locked P&L ($M)': round(locked,2), 'Open P&L ($M)': round(open_,2),
        'Commodity Hedge': benchmark, 'Freight Hedge': 'None — gap',
    })

hedge_df = pd.DataFrame(hedge_rows)
print('=== HEDGE DASHBOARD ===')
print(hedge_df.to_string(index=False))
print(f"\nTotal Locked: ${hedge_df['Locked P&L ($M)'].sum():.2f}M   Open: ${hedge_df['Open P&L ($M)'].sum():.2f}M")
print('\nNOTE: Freight exposure is UNHEDGED. OU freight (σ=0.35) contributes ~8-15% of P&L variance.')
print('Priority candidates: Ships 2, 5, 6 (flexible FOB). Instrument: FFA or LNG freight swap.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
ax = axes[0]
x = np.arange(len(hedge_df)); w = 0.40
b1 = ax.bar(x-w/2, hedge_df['Locked P&L ($M)'], w, label='Locked', color='#2980b9', alpha=0.9)
b2 = ax.bar(x+w/2, hedge_df['Open P&L ($M)'],   w, label='Open',   color='#e67e22', alpha=0.9)
for bars in [b1, b2]:
    for bar in bars:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
                f'${bar.get_height():.1f}M', ha='center', va='bottom', fontsize=7.5)
ax.set_xticks(x); ax.set_xticklabels(hedge_df['Ship'])
ax.set_ylabel('P&L ($M per cargo)'); ax.set_title('Locked vs Open P&L', fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.25)

ax2 = axes[1]
cats = ['Committed','Swing']
lk = hedge_df.groupby('Category')['Locked P&L ($M)'].sum()
op = hedge_df.groupby('Category')['Open P&L ($M)'].sum()
x2 = np.arange(len(cats))
ax2.bar(x2-w/2, [lk.get(c,0) for c in cats], w, label='Locked', color='#2980b9', alpha=0.9)
ax2.bar(x2+w/2, [op.get(c,0) for c in cats], w, label='Open',   color='#e67e22', alpha=0.9)
ax2.set_xticks(x2); ax2.set_xticklabels(cats)
ax2.set_ylabel('P&L ($M)'); ax2.set_title('By category', fontweight='bold')
ax2.legend(); ax2.grid(axis='y', alpha=0.25)
plt.suptitle('Hedge Dashboard', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

---
## Step 6 — Stochastic Price Simulation
**Price factors:** correlated GBM for Brent, HH, TTF, JKM  
**Freight:** Ornstein-Uhlenbeck (mean-reverting, gas-linked)

In [ ]:
N_SCENARIOS  = 2000
HORIZON_DAYS = 90
DT           = 1 / 252

S0     = np.array([BRENT, HH, TTF, JKM])
SIGMA  = np.array([0.28, 0.45, 0.35, 0.30])
CORR   = np.array([[1.00,0.15,0.40,0.35],[0.15,1.00,0.25,0.20],
                   [0.40,0.25,1.00,0.75],[0.35,0.20,0.75,1.00]])
CHOL   = np.linalg.cholesky(np.diag(SIGMA) @ CORR @ np.diag(SIGMA))

Z_p    = np.random.standard_normal((HORIZON_DAYS, N_SCENARIOS, 4))
dW_p   = Z_p @ CHOL.T * np.sqrt(DT)
lp     = np.cumsum((np.zeros(4) - 0.5*SIGMA**2)*DT + dW_p, axis=0)
price_paths = np.concatenate([S0[np.newaxis,np.newaxis,:]*np.ones((1,N_SCENARIOS,1)),
                               S0*np.exp(lp)], axis=0)

# ── Freight: OU with TTF/JKM linkage ─────────────────────────────────────────
KAPPA = 3.0; THETA = 1.0; F_SIGMA = 0.35
L_TTF = 0.30; L_JKM = 0.35

freight_paths = np.zeros((HORIZON_DAYS+1, N_SCENARIOS))
freight_paths[0] = BASE_FREIGHT_INDEX
Z_f = np.random.standard_normal((HORIZON_DAYS, N_SCENARIOS))
idio_w = np.sqrt(max(1e-8, 1 - L_TTF**2 - L_JKM**2))

for t in range(HORIZON_DAYS):
    x = freight_paths[t]
    shock = L_TTF*Z_p[t,:,2] + L_JKM*Z_p[t,:,3] + idio_w*Z_f[t]
    freight_paths[t+1] = np.maximum(0.2, x + KAPPA*(THETA-x)*DT + F_SIGMA*np.sqrt(DT)*shock)

print(f'Simulated {N_SCENARIOS:,} scenarios × {HORIZON_DAYS} days')
print(f'Price GBM | Freight OU (κ={KAPPA}, θ={THETA}, σ={F_SIGMA}, half-life≈{np.log(2)/KAPPA*252:.0f} days)')
print(f'Final means: Brent={price_paths[-1,:,0].mean():.1f}  HH={price_paths[-1,:,1].mean():.2f}',
      f'TTF={price_paths[-1,:,2].mean():.2f}  JKM={price_paths[-1,:,3].mean():.2f}',
      f'Freight={freight_paths[-1].mean():.3f}')

In [ ]:
t = np.arange(price_paths.shape[0])
colors_p = ['#2c3e50','#e74c3c','#2980b9','#27ae60']
labels_p = ['Brent ($/bbl)','HH ($/MMBtu)','TTF ($/MMBtu)','JKM ($/MMBtu)']

fig = plt.figure(figsize=(14,9))
gs  = gridspec.GridSpec(2, 3, figure=fig)

for i in range(4):
    ax = fig.add_subplot(gs[i//3, i%3])
    p5,p25,p50,p75,p95 = np.percentile(price_paths[:,:,i],[5,25,50,75,95],axis=1)
    ax.fill_between(t,p5,p95,alpha=0.12,color=colors_p[i])
    ax.fill_between(t,p25,p75,alpha=0.22,color=colors_p[i])
    ax.plot(t,p50,color=colors_p[i],lw=2,label='Median')
    ax.axhline(S0[i],color='grey',lw=1.2,ls='--',label='Spot')
    ax.set_title(labels_p[i],fontweight='bold',fontsize=10)
    ax.legend(fontsize=8); ax.grid(alpha=0.25)

ax_f = fig.add_subplot(gs[1,1])
fp5,fp25,fp50,fp75,fp95 = np.percentile(freight_paths,[5,25,50,75,95],axis=1)
ax_f.fill_between(t,fp5,fp95,alpha=0.12,color='#8e44ad')
ax_f.fill_between(t,fp25,fp75,alpha=0.22,color='#8e44ad')
ax_f.plot(t,fp50,color='#8e44ad',lw=2,label='Median (OU)')
ax_f.axhline(1.0,color='grey',lw=1.2,ls='--'); ax_f.set_title('Freight Index (OU)',fontweight='bold',fontsize=10)
ax_f.legend(fontsize=8); ax_f.grid(alpha=0.25)

ax_c = fig.add_subplot(gs[1,2])
gbm_f = np.zeros((HORIZON_DAYS+1,N_SCENARIOS)); gbm_f[0]=1.0
Z_g = np.random.standard_normal((HORIZON_DAYS,N_SCENARIOS))
gbm_f[1:] = np.exp(np.cumsum((-0.5*0.55**2)*DT + 0.55*np.sqrt(DT)*Z_g,axis=0))
for pa,lb,col in [(freight_paths,'OU (σ=0.35)','#8e44ad'),(gbm_f,'GBM (σ=0.55)','#e74c3c')]:
    p5_,p95_ = np.percentile(pa,[5,95],axis=1)
    ax_c.fill_between(t,p5_,p95_,alpha=0.12,color=col)
    ax_c.plot(t,np.percentile(pa,50,axis=1),color=col,lw=1.8,label=lb)
ax_c.set_title('Freight: OU vs GBM',fontweight='bold',fontsize=10)
ax_c.legend(fontsize=8); ax_c.grid(alpha=0.25)

plt.suptitle(f'Stochastic Simulation — {N_SCENARIOS:,} Scenarios, {HORIZON_DAYS}-day Horizon',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

---
## Step 7 — Portfolio P&L Under Stochastic Scenarios

In [ ]:
BRENT_S = price_paths[-1,:,0]; HH_S = price_paths[-1,:,1]
TTF_S   = price_paths[-1,:,2]; JKM_S = price_paths[-1,:,3]
FREIGHT_S = freight_paths[-1]

def ship_pnl(ship_row, brent_s, hh_s, ttf_s, jkm_s, freight_s):
    phys   = route_margin(ship_row['Best Origin'], ship_row['Best Dest'],
                          contract=ship_row['Contract'],
                          brent=brent_s, hh=hh_s, ttf=ttf_s, jkm=jkm_s,
                          freight_index=freight_s)
    locked = ship_row['Base Margin']
    hr     = ship_row['Hedge Ratio']
    return (hr*locked + (1-hr)*phys) * CARGO_SIZE_MMBTU / 1e6

pnl_df    = pd.DataFrame({s['Ship']: ship_pnl(s,BRENT_S,HH_S,TTF_S,JKM_S,FREIGHT_S)
                          for _,s in fleet.iterrows()})
fleet_pnl = pnl_df.sum(axis=1)

def stats(pnl):
    p5 = np.percentile(pnl,5)
    return {'E[P&L]':pnl.mean(),'P5':p5,'P95':np.percentile(pnl,95),
            'CVaR':pnl[pnl<=p5].mean(),'Std':pnl.std()}

s = stats(fleet_pnl)
print(f'Expected P&L: ${s["E[P&L]"]:.2f}M  |  P5: ${s["P5"]:.2f}M  |  P95: ${s["P95"]:.2f}M  |  CVaR: ${s["CVaR"]:.2f}M')

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(14,5))
ax = axes[0]
ax.hist(fleet_pnl,bins=80,color='#2980b9',alpha=0.82,edgecolor='white')
ax.axvline(s['E[P&L]'],color='#27ae60',lw=2.2,label=f"E[P&L]=${s['E[P&L]']:.2f}M")
ax.axvline(s['P5'],color='#e74c3c',lw=2.2,ls='--',label=f"P5=${s['P5']:.2f}M")
ax.axvline(s['P95'],color='#9b59b6',lw=2.2,ls='--',label=f"P95=${s['P95']:.2f}M")
ax.axvline(s['CVaR'],color='#c0392b',lw=2.2,ls=':',label=f"CVaR=${s['CVaR']:.2f}M")
ax.set_xlabel('Fleet P&L ($M)'); ax.set_ylabel('Frequency')
ax.set_title('Fleet P&L Distribution',fontweight='bold')
ax.legend(fontsize=9); ax.grid(alpha=0.25)

ax2 = axes[1]
ev=pnl_df.mean(); p5=pnl_df.quantile(0.05); p95=pnl_df.quantile(0.95)
xpos=np.arange(len(ev))
cat_c=[colors[fleet.loc[fleet['Ship']==s,'Category'].values[0]] for s in ev.index]
ax2.bar(xpos,ev,color=cat_c,alpha=0.85)
ax2.errorbar(xpos,ev,yerr=[ev-p5,p95-ev],fmt='none',color='#2c3e50',capsize=5,lw=1.8)
ax2.set_xticks(xpos); ax2.set_xticklabels(ev.index,rotation=15)
ax2.set_ylabel('E[P&L] ($M)'); ax2.set_title('Per-ship E[P&L] with P5-P95',fontweight='bold')
ax2.grid(axis='y',alpha=0.25)
from matplotlib.patches import Patch
ax2.legend(handles=[Patch(color=v,label=k) for k,v in colors.items()],fontsize=9)
plt.suptitle('Stochastic Scenario Results',fontsize=13,fontweight='bold')
plt.tight_layout(); plt.show()

---
## Step 8 — Hedge Ratio Sensitivity (Tiered by Bucket)

In [ ]:
BUCKET_BANDS = {'Committed':(0.70,1.00),'Swing':(0.30,0.60),'Cover':(0.00,0.30)}
sweep_t = np.linspace(0,1,21)
results = []

for t_val in sweep_t:
    total_pnl = np.zeros(N_SCENARIOS)
    for _,ship in fleet.iterrows():
        lo,hi = BUCKET_BANDS[ship['Category']]
        hr    = lo + t_val*(hi-lo)
        phys  = route_margin(ship['Best Origin'],ship['Best Dest'],contract=ship['Contract'],
                             brent=BRENT_S,hh=HH_S,ttf=TTF_S,jkm=JKM_S,freight_index=FREIGHT_S)
        total_pnl += (hr*ship['Base Margin']+(1-hr)*phys)*CARGO_SIZE_MMBTU/1e6
    p5_ = np.percentile(total_pnl,5)
    results.append({
        'Committed HR': round(BUCKET_BANDS['Committed'][0]+t_val*(BUCKET_BANDS['Committed'][1]-BUCKET_BANDS['Committed'][0]),2),
        'Swing HR':     round(BUCKET_BANDS['Swing'][0]+t_val*(BUCKET_BANDS['Swing'][1]-BUCKET_BANDS['Swing'][0]),2),
        'E[P&L]':total_pnl.mean(),'P5':p5_,'CVaR':total_pnl[total_pnl<=p5_].mean(),'Vol':total_pnl.std()})

strat_df = pd.DataFrame(results)

fig,axes = plt.subplots(1,2,figsize=(14,5))
ax=axes[0]
ax.plot(strat_df['Committed HR'],strat_df['E[P&L]'],'o-',color='#2980b9',lw=2,label='E[P&L]')
ax.fill_between(strat_df['Committed HR'],strat_df['P5'],strat_df['E[P&L]'],alpha=0.15,color='#2980b9')
ax.plot(strat_df['Committed HR'],strat_df['P5'],'--',color='#e74c3c',lw=1.5,label='P5')
ax.plot(strat_df['Committed HR'],strat_df['CVaR'],':',color='#c0392b',lw=1.8,label='CVaR')
ax.set_xlabel('Committed HR (Swing scales 30-60%)'); ax.set_ylabel('P&L ($M)')
ax.set_title('P&L vs Tiered Hedge Sweep',fontweight='bold')
ax.legend(); ax.grid(alpha=0.25)

ax2=axes[1]
ax2.plot(strat_df['Vol'],strat_df['E[P&L]'],'o-',color='#27ae60',lw=2)
for _,row in strat_df.iloc[::4].iterrows():
    ax2.annotate(f"C:{row['Committed HR']:.0%}\nS:{row['Swing HR']:.0%}",
                 (row['Vol'],row['E[P&L]']),textcoords='offset points',xytext=(6,4),fontsize=7.5)
ax2.set_xlabel('Volatility ($M)'); ax2.set_ylabel('E[P&L] ($M)')
ax2.set_title('Risk-Return Frontier',fontweight='bold'); ax2.grid(alpha=0.25)
plt.suptitle('Tiered Hedge Strategy Comparison',fontsize=13,fontweight='bold')
plt.tight_layout(); plt.show()

---
## Step 9 — Two-Stage Optimization

In [ ]:
def two_stage_pnl(fleet_df, brent_s, hh_s, ttf_s, jkm_s, freight_s):
    total = np.zeros(len(brent_s))
    for _,ship in fleet_df.iterrows():
        orig=ship['Best Origin']; hr=ship['Hedge Ratio']
        contract=ship['Contract']; flexible=ship['Flexible Destinations']
        m_eu = route_margin(orig,'Europe',contract=contract,brent=brent_s,hh=hh_s,ttf=ttf_s,jkm=jkm_s,freight_index=freight_s)
        m_as = route_margin(orig,'Asia',  contract=contract,brent=brent_s,hh=hh_s,ttf=ttf_s,jkm=jkm_s,freight_index=freight_s)
        if contract=='DES' or not flexible:
            phys = m_eu if ship['Best Dest']=='Europe' else m_as
        else:
            phys = np.maximum(m_eu, m_as)
        total += (hr*ship['Base Margin']+(1-hr)*phys)*CARGO_SIZE_MMBTU/1e6
    return total

pnl_fixed  = fleet_pnl
pnl_2stage = two_stage_pnl(fleet,BRENT_S,HH_S,TTF_S,JKM_S,FREIGHT_S)
s_fix = stats(pnl_fixed); s_2st = stats(pnl_2stage)

print(f'{"Strategy":<20} {"E[P&L]":>10} {"P5":>10} {"CVaR":>10} {"P95":>10}')
print('-'*55)
print(f'{"Fixed routes":<20} ${s_fix["E[P&L]"]:>8.2f}M ${s_fix["P5"]:>7.2f}M ${s_fix["CVaR"]:>7.2f}M ${s_fix["P95"]:>7.2f}M')
print(f'{"Two-stage":<20} ${s_2st["E[P&L]"]:>8.2f}M ${s_2st["P5"]:>7.2f}M ${s_2st["CVaR"]:>7.2f}M ${s_2st["P95"]:>7.2f}M')
print(f'\nOptimality uplift: +${s_2st["E[P&L]"]-s_fix["E[P&L]"]:.2f}M')

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(14,5))
ax=axes[0]
bins=np.linspace(min(pnl_fixed.min(),pnl_2stage.min()),max(pnl_fixed.max(),pnl_2stage.max()),80)
ax.hist(pnl_fixed, bins=bins,alpha=0.65,color='#e74c3c',label='Fixed routes')
ax.hist(pnl_2stage,bins=bins,alpha=0.65,color='#27ae60',label='Two-stage')
ax.axvline(s_fix['E[P&L]'],color='#c0392b',lw=2.2,ls='--')
ax.axvline(s_2st['E[P&L]'],color='#1e8449',lw=2.2,ls='--')
ax.set_xlabel('Fleet P&L ($M)'); ax.set_ylabel('Count')
ax.set_title('Fixed vs Two-Stage',fontweight='bold')
ax.legend(fontsize=10); ax.grid(alpha=0.25)

ax2=axes[1]
spread=JKM_S-TTF_S; opt_val=pnl_2stage-pnl_fixed
ax2.scatter(spread,opt_val,alpha=0.15,s=6,color='#2980b9')
si=np.argsort(spread); w=200
ax2.plot(np.convolve(spread[si],np.ones(w)/w,'valid'),
         np.convolve(opt_val[si],np.ones(w)/w,'valid'),color='#e74c3c',lw=2.2,label='Median')
ax2.axhline(0,color='grey',lw=1,ls='--'); ax2.axvline(0,color='grey',lw=1,ls='--')
ax2.set_xlabel('JKM−TTF spread ($/MMBtu)'); ax2.set_ylabel('Optionality value ($M)')
ax2.set_title('When does optionality pay off?',fontweight='bold')
ax2.legend(fontsize=9); ax2.grid(alpha=0.25)
plt.suptitle('Two-Stage Optimization',fontsize=13,fontweight='bold')
plt.tight_layout(); plt.show()

---
## Step 10 — Route Decision & Contract Status Dashboard

In [ ]:
decision_rows = []
for _,ship in fleet.iterrows():
    orig=ship['Best Origin']; cd=ship['Best Dest']; ct=ship['Contract']
    cm=route_margin(orig,cd,contract=ct)
    ad='Asia' if cd=='Europe' else 'Europe'
    am=route_margin(orig,ad,contract=ct)
    can_sw = ct=='FOB' and ship['Flexible Destinations']
    decision_rows.append({
        'Ship':ship['Ship'],'Contract':ct,'Flexible':ship['Flexible Destinations'],
        'Origin':orig,'Chosen':cd,'Chosen M':round(cm,3),
        'Alt':ad,'Alt M':round(am,3),
        'Gap':round(abs(cm-am),3),'Can Switch':can_sw})

print('=== ROUTE DECISION EXPLANATION ===')
print(pd.DataFrame(decision_rows).to_string(index=False))

print('\n=== CONTRACT STATUS DASHBOARD ===')
status = fleet[['Ship','Contract','Category','Flexible Destinations',
                'Best Origin','Best Dest','Base Margin','Hedge Ratio',
                'Effective Option Premium']].copy()
status['Exposure'] = status.apply(
    lambda r: ('HH+' if r['Best Origin']=='US' else 'Brent+') +
              ('TTF' if r['Best Dest']=='Europe' else 'JKM'), axis=1)
print(status.to_string(index=False))

---
## Step 11 — Final Desk-Ready Summary

In [ ]:
W=78
print('='*W)
print('  LNG FLEET PORTFOLIO OPTIMIZATION — FINAL SUMMARY')
print('='*W)

print('\n[1] BASE-CASE ROUTE VIEW')
print(route_matrix[['Origin','Destination','Total Cost','FOB Margin','DES Margin',
                     'FOB Margin/day','DES Margin/day']].to_string(index=False))

print('\n[2] HEDGE DASHBOARD')
print(hedge_df[['Ship','Category','Contract','Hedge Ratio',
                'Locked P&L ($M)','Open P&L ($M)','Commodity Hedge']].to_string(index=False))
print(f"    Total Locked: ${hedge_df['Locked P&L ($M)'].sum():.2f}M   Open: ${hedge_df['Open P&L ($M)'].sum():.2f}M")

print(f'\n[3] SCENARIO SUMMARY ({N_SCENARIOS:,} paths, {HORIZON_DAYS}-day horizon)')
print(f'    Fixed routes — E[P&L]: ${s_fix["E[P&L]"]:.2f}M  P5: ${s_fix["P5"]:.2f}M  CVaR: ${s_fix["CVaR"]:.2f}M  P95: ${s_fix["P95"]:.2f}M')
print(f'    Two-stage    — E[P&L]: ${s_2st["E[P&L]"]:.2f}M  P5: ${s_2st["P5"]:.2f}M  CVaR: ${s_2st["CVaR"]:.2f}M  P95: ${s_2st["P95"]:.2f}M')
print(f'    Optimality uplift: +${s_2st["E[P&L]"]-s_fix["E[P&L]"]:.2f}M')

print('\n[4] SHIP ACTION LIST')
for _,ship in fleet.iterrows():
    print(f'    {ship["Ship"]:8s}  ({ship["Contract"]}, {ship["Category"]:10s})  {ship["Action"]}')

print('\n[5] OPEN RISK GAPS')
print(f'    Freight: UNHEDGED. OU freight (κ={KAPPA}, σ={F_SIGMA}) ~8-15% of P&L variance.')
print('    Priority: Ships 2, 5, 6. Instrument: FFA or LNG freight swap.')

print('\n[6] MODEL ASSUMPTIONS')
print('    · Boil-off at fixed $12/MMBtu (not dest-linked)')
print(f'    · Freight: OU linked to TTF/JKM, κ={KAPPA}, θ={THETA}, σ={F_SIGMA}')
print('    · Hedge sweep: tiered by bucket (committed 70-100%, swing 30-60%)')
print('    · Single cargo per ship per cycle')

print('\n'+'='*W)
print(f'  BOTTOM LINE: +${s_2st["E[P&L]"]-s_fix["E[P&L]"]:.2f}M from swing-ship optionality.')
print('  Reward optionality explicitly, not as a residual.')
print('='*W)

---
## Governance

| Trigger | Action |
|---|---|
| Daily | Refresh benchmarks and deterministic matrix |
| Price move >5% | Re-run CVaR, re-classify buckets |
| Weekly | Re-run stochastic model, recalibrate OU freight |
| Major shock | Full re-run, assess swing-ship reassignment |
| New commitment | Update fleet table, re-optimise |

---
*LNG Fleet Portfolio Optimization · March 2026*